# IBM HR Employee Attrition 

**Dataset:** IBM HR Analytics Employee Attrition and Performance  
**Goal:** Predict which employees are likely to leave the company  

## Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    confusion_matrix, accuracy_score, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve, f1_score
)
from imblearn.over_sampling import SMOTE
RANDOM_STATE = 42  

# Set seaborn style for clean, professional plots
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')
import joblib


## Load & Inspect Data

In [ ]:
from pathlib import Path

local_path = Path("IBM-HR-Analytics-Employee-Attrition-and-Performance.csv")
kaggle_path = Path("/kaggle/input/datasets/aryanmdev/tech-hiring-and-la/IBM-HR-Analytics-Employee-Attrition-and-Performance.csv")
data_path = local_path if local_path.exists() else kaggle_path

df = pd.read_csv(data_path)
df_raw = df.copy()

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
missing = df.isnull().sum()

if missing.sum() == 0:
    print("No missing values")
else:
    print(missing[missing > 0])

In [ ]:
counts = df['Attrition'].value_counts()

print(counts)
print(f"Ratio (No/Yes): {counts['No'] / counts['Yes']:.1f}")

sns.countplot(data=df, x='Attrition')
plt.show()

**Observation**
The dataset is heavily imbalanced : ~84% employees stay (No) vs ~16% leave (Yes). 

## Data Preprocessing

In [ ]:
cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
df = df.drop(columns=cols, errors='ignore')

print(df.shape)

In [ ]:
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
print(df['Attrition'].value_counts())

In [ ]:
ordinal_cols = [
    'Education', 'EnvironmentSatisfaction', 'JobInvolvement',
    'JobSatisfaction', 'PerformanceRating', 'RelationshipSatisfaction',
    'WorkLifeBalance', 'JobLevel', 'StockOptionLevel'
]

print(df[ordinal_cols].nunique())

In [ ]:
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['OverTime'] = df['OverTime'].map({'Yes': 1, 'No': 0})

print(df[['Gender', 'OverTime']].value_counts())

In [ ]:
nominal_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print(df.shape)

In [ ]:
print(df.shape)
print(df.dtypes)

## Outlier Detection

We use the **IQR (Interquartile Range) method** to detect outliers:
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 - Q1
- Outliers are values below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR`

We'll **flag** outliers but NOT remove them — some "outliers" might be real high-earners or long-tenured employees.

In [ ]:
cols = ['MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany', 'DailyRate', 'Age']

df[cols].plot(
    kind='box',
    subplots=True,
    layout=(2, 3),  
    figsize=(12, 8)  
)

plt.tight_layout()
plt.show()

In [ ]:
cols = ['MonthlyIncome', 'TotalWorkingYears', 'YearsAtCompany', 'DailyRate', 'Age']

Q1 = df[cols].quantile(0.25)
Q3 = df[cols].quantile(0.75)
IQR = Q3 - Q1

outliers = ((df[cols] < (Q1 - 1.5 * IQR)) | (df[cols] > (Q3 + 1.5 * IQR))).sum()

print(outliers)

### Outlier Decision
- **MonthlyIncome** and **TotalWorkingYears** have noticeable outliers (senior employees / executives)
- These are **valid data points**, not errors — removing them would bias the model
- Tree-based models (XGBoost, CatBoost) are naturally robust to outliers
- For linear models, StandardScaler helps reduce outlier impact

## Exploratory Data Analysis (EDA)

In [ ]:
def plot_attrition(data, col, numeric=False):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # LEFT: distribution
    if numeric:
        sns.histplot(data=data, x=col, hue='Attrition', ax=axes[0])
    else:
        sns.countplot(data=data, x=col, ax=axes[0])
        axes[0].tick_params(axis='x', rotation=45)

    # RIGHT: attrition rate
    if numeric:
        bins = pd.qcut(data[col], q=5, duplicates='drop')
        rate = data.groupby(bins)['Attrition'].mean()
    else:
        rate = data.groupby(col)['Attrition'].mean()

    rate.plot(kind='bar', ax=axes[1])
    axes[1].set_ylabel('Attrition Rate')

    plt.tight_layout()
    plt.show()

In [ ]:
# We use df_raw for readable EDA charts (before encoding)
# But drop the useless columns
df_eda = df_raw.drop(columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'], errors='ignore')
df_eda['Attrition'] = df_eda['Attrition'].map({'Yes': 1, 'No': 0})

In [ ]:
plot_attrition(df_eda, 'Age', numeric=True)

In [ ]:
plot_attrition(df_eda, 'Gender')

In [ ]:
plot_attrition(df_eda, 'Department')

In [ ]:
plot_attrition(df_eda, 'OverTime')

In [ ]:
plot_attrition(df_eda, 'MaritalStatus')

In [ ]:
plot_attrition(df_eda, 'JobRole')

In [ ]:
plot_attrition(df_eda, 'BusinessTravel')

In [ ]:
plot_attrition(df_eda, 'MonthlyIncome', numeric=True)

In [ ]:
plot_attrition(df_eda, 'YearsAtCompany', numeric=True)

In [ ]:
plot_attrition(df_eda, 'JobSatisfaction')

In [ ]:
plot_attrition(df_eda, 'WorkLifeBalance')

In [ ]:
corr = df.corr()

sns.heatmap(corr, cmap='coolwarm', center=0)
plt.show()

In [ ]:
corr = df.corr()['Attrition'].drop('Attrition').sort_values(key=abs, ascending=False)

print(corr.head(10))

In [ ]:
cols = corr.head(5).index

for col in cols:
    sns.boxplot(x='Attrition', y=col, data=df)
    plt.title(col)
    plt.show()

## Statistical Analysis (ANOVA & Chi-Square Tests)

In [ ]:
from scipy.stats import f_oneway

numerical_cols = df_eda.select_dtypes(include=[np.number]).columns.drop('Attrition')

anova_results = []
for col in numerical_cols:
    # Split feature values by Attrition group
    group_stay = df_eda[df_eda['Attrition'] == 0][col]
    group_leave = df_eda[df_eda['Attrition'] == 1][col]
    
    # Run one-way ANOVA
    f_score, p_value = f_oneway(group_stay, group_leave)
    anova_results.append({'Feature': col, 'F_Score': f_score, 'P_Value': p_value})

anova_df = pd.DataFrame(anova_results).sort_values('F_Score', ascending=False)

print("ANOVA Test Results (Numerical Features vs Attrition)")
print("=" * 65)
for _, row in anova_df.iterrows():
    sig = "\u2705 Significant" if row['P_Value'] < 0.05 else "\u274c Not Significant"
    print(f"  {row['Feature']:30s}  F={row['F_Score']:8.2f}  p={row['P_Value']:.6f}  {sig}")

In [ ]:
fig, ax = plt.subplots()

colors = ['green' if p < 0.05 else 'red' for p in anova_df['P_Value']]
ax.barh(anova_df['Feature'], anova_df['F_Score'], color=colors)

plt.show()

In [ ]:
from scipy.stats import chi2_contingency

categorical_test_cols = ['BusinessTravel', 'Department', 'EducationField', 'Gender',
                         'JobRole', 'MaritalStatus', 'OverTime']

chi2_results = []
for col in categorical_test_cols:
    contingency = pd.crosstab(df_eda[col], df_eda['Attrition'])
    chi2_stat, p_value, dof, expected = chi2_contingency(contingency)
    chi2_results.append({'Feature': col, 'Chi2_Stat': chi2_stat, 'P_Value': p_value, 'DoF': dof})

chi2_df = pd.DataFrame(chi2_results).sort_values('Chi2_Stat', ascending=False)

print(chi2_df)

In [ ]:
fig, ax = plt.subplots()

colors = ['green' if p < 0.05 else 'red' for p in chi2_df['P_Value']]
ax.barh(chi2_df['Feature'], chi2_df['Chi2_Stat'], color=colors)

plt.show()

## Feature Engineering (New Columns)

In [ ]:
df['IncomePerYear'] = df['MonthlyIncome'] / (df['TotalWorkingYears'] + 1)

df['PromotionLag'] = df['YearsAtCompany'] - df['YearsSinceLastPromotion']

df['TenureRatio'] = df['YearsInCurrentRole'] / (df['YearsAtCompany'] + 1)

df['SatisfactionScore'] = (
    df['JobSatisfaction'] + df['EnvironmentSatisfaction'] + df['WorkLifeBalance']
) / 3

df['IsOverworked'] = ((df['OverTime'] == 1) & (df['WorkLifeBalance'] <= 2)).astype(int)

df['ExperienceLevel'] = pd.cut(
    df['TotalWorkingYears'],
    bins=[0, 2, 5, 10, 20, 50],
    labels=[0, 1, 2, 3, 4],
    include_lowest=True
).astype(int)

print(df.shape)

In [ ]:
new_features = ['IncomePerYear', 'PromotionLag', 'TenureRatio',
                'SatisfactionScore', 'IsOverworked', 'ExperienceLevel']

new_corr = df[new_features].corrwith(df['Attrition'])

print(new_corr)

fig, ax = plt.subplots()
colors = ['red' if v > 0 else 'green' for v in new_corr.values]
new_corr.plot(kind='barh', color=colors, ax=ax)

plt.show()

## Train/Test Split

I split the data 80/20 and use stratification to ensure both train and test sets  
have the same proportion of attrition (Yes/No) crucial for imbalanced datasets.

In [ ]:
X = df.drop(columns=['Attrition'])
y = df['Attrition']

print(f"Features (X): {X.shape}")
print(f"Target  (y): {y.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(X_train_scaled.shape, X_test_scaled.shape)

## Cross-Validation Setup
  
**5-Fold Stratified Cross-Validation** trains 5 models on different splits and averages the results,  


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def cross_val_report(model, X, y, model_name):
    scoring = ['accuracy', 'f1', 'roc_auc']
    cv_results = cross_validate(model, X, y, cv=cv_strategy,
                                scoring=scoring, return_train_score=False)
    
    result_dict = {}
    for metric in scoring:
        scores = cv_results[f'test_{metric}']
        print(metric, scores.mean(), scores.std())
        result_dict[metric] = scores.mean()
    
    return result_dict

## Baseline Model: Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=RANDOM_STATE
)

lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print(accuracy_score(y_test, y_pred_lr))
print(roc_auc_score(y_test, y_prob_lr))
print(f1_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

In [ ]:
lr_cv_scores = cross_val_report(
    LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    X_train_scaled, y_train, "Logistic Regression"
)

In [ ]:
# ===== CONFUSION MATRIX HEATMAP =====
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Stay (0)', 'Leave (1)'],
            yticklabels=['Stay (0)', 'Leave (1)'])
ax.set_title('Logistic Regression — Confusion Matrix', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(fpr_lr, tpr_lr, color='#3498db', linewidth=2, label=f'LR (AUC = {auc_lr:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.500)')
ax1.set_title('ROC Curve — Logistic Regression', fontweight='bold')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.legend()
ax1.grid(True, alpha=0.3)


precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_prob_lr)
ax2.plot(recall_lr, precision_lr, color='#e74c3c', linewidth=2)
ax2.set_title('Precision-Recall Curve — Logistic Regression', fontweight='bold')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Class Imbalance: Detection & Fix

In [ ]:
# ===== 9A: PROVE THE PROBLEM WITH A DUMMY CLASSIFIER =====
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train_scaled, y_train)
y_pred_dummy = dummy.predict(X_test_scaled)

dummy_acc = accuracy_score(y_test, y_pred_dummy)
print(f"Dummy Classifier (always predicts 'No'):")
print(f"  Accuracy: {dummy_acc:.4f} ({dummy_acc*100:.1f}%)")
print(f"  F1 Score: {f1_score(y_test, y_pred_dummy):.4f}")
print(f"\n→ The dummy gets {dummy_acc*100:.1f}% accuracy by learning NOTHING!")
print(f"→ Our LR model must beat this to be useful.")

In [ ]:
# ===== CLASS DISTRIBUTION PIE CHART =====
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Training set
y_train.value_counts().plot.pie(
    labels=['Stay (No)', 'Leave (Yes)'],
    autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'],
    ax=ax1, startangle=90
)
ax1.set_title('Training Set — Class Distribution', fontweight='bold')
ax1.set_ylabel('')

# After we'll show SMOTE — for now just show the imbalance
ax2.text(0.5, 0.5, 'After SMOTE\n(Next Step)', ha='center', va='center',
         fontsize=16, fontweight='bold', transform=ax2.transAxes)
ax2.set_title('After SMOTE — Balanced', fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

### Fixing with SMOTE

**SMOTE (Synthetic Minority Over-sampling Technique)** creates **synthetic** samples  
of the minority class — it doesn't just duplicate existing rows!

It works by:
1. Picking a minority sample
2. Finding its k nearest neighbors (also minority)
3. Creating a new point somewhere between them

This gives the model more diverse examples to learn from.

In [ ]:
# ===== 9B: APPLY SMOTE TO BALANCE THE TRAINING DATA =====
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:")
print(f"  Class 0 (Stay):  {(y_train == 0).sum()}")
print(f"  Class 1 (Leave): {(y_train == 1).sum()}")

print("\nAfter SMOTE:")
print(f"  Class 0 (Stay):  {(y_train_sm == 0).sum()}")
print(f"  Class 1 (Leave): {(y_train_sm == 1).sum()}")

print(f"\n✅ Training set is now balanced! ({len(y_train_sm)} total samples)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))


pd.Series(y_train).value_counts().plot.pie(
    labels=['Stay', 'Leave'], autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'], ax=ax1, startangle=90
)
ax1.set_title('Before SMOTE', fontweight='bold')
ax1.set_ylabel('')


pd.Series(y_train_sm).value_counts().plot.pie(
    labels=['Stay', 'Leave'], autopct='%1.1f%%',
    colors=['#2ecc71', '#e74c3c'], ax=ax2, startangle=90
)
ax2.set_title('After SMOTE', fontweight='bold')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

## XGBoost Model

XGBoost is a **gradient boosting** algorithm — it builds trees sequentially,  
where each new tree corrects the errors of the previous ones.

We use `scale_pos_weight` to handle class imbalance directly,  
plus manually tuned hyperparameters (not defaults).

In [ ]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Class imbalance ratio (No/Yes): {ratio:.2f}")
print(f"→ We'll give {ratio:.1f}x more weight to 'Yes' (attrition) samples")

### Hyperparameter Tuning with RandomizedSearchCV

Instead of guessing parameters, we let **RandomizedSearchCV** search over a grid  
of hyperparameter combinations using cross-validation to find the best ones.

In [ ]:
# ===== HYPERPARAMETER TUNING FOR XGBOOST =====
from sklearn.model_selection import RandomizedSearchCV

xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
}

xgb_base = XGBClassifier(
    scale_pos_weight=ratio,
    eval_metric='logloss',
    random_state=RANDOM_STATE

)

xgb_search = RandomizedSearchCV(
    xgb_base,
    param_distributions=xgb_param_grid,
    n_iter=50,
    scoring='f1',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
xgb_search.fit(X_train_scaled, y_train)

print(f"\n\u2705 Best F1 Score from CV: {xgb_search.best_score_:.4f}")
print(f"Best Parameters:")
for param, value in xgb_search.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
# ===== TRAIN XGBOOST WITH BEST PARAMETERS =====
xgb_model = xgb_search.best_estimator_

y_pred_xgb = xgb_model.predict(X_test_scaled)
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

print("XGBoost (Tuned) \u2014 Test Results")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_xgb):.4f}")
print(f"F1 (Yes):  {f1_score(y_test, y_pred_xgb):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_xgb)}")

In [ ]:
# ===== CROSS-VALIDATE XGBOOST =====
xgb_cv_scores = cross_val_report(
    xgb_search.best_estimator_,
    X_train_scaled, y_train, "XGBoost (Tuned)"
)

In [ ]:
# ===== XGBOOST CONFUSION MATRIX =====
fig, ax = plt.subplots(figsize=(6, 5))
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=['Stay (0)', 'Leave (1)'],
            yticklabels=['Stay (0)', 'Leave (1)'])
ax.set_title('XGBoost — Confusion Matrix', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ===== XGBOOST FEATURE IMPORTANCE (Top 15) =====
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feature_importance['feature'], feature_importance['importance'], color='#e67e22')
ax.set_title('XGBoost — Top 15 Feature Importances', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# ===== XGBOOST ROC CURVE =====
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
auc_xgb = roc_auc_score(y_test, y_prob_xgb)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_xgb, tpr_xgb, color='#e67e22', linewidth=2, label=f'XGBoost (AUC = {auc_xgb:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_title('ROC Curve — XGBoost', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## CatBoost Model

CatBoost (Categorical Boosting) is another gradient boosting algorithm from Yandex.  
It handles categorical features natively and often works well with minimal tuning.  
We use `class_weights` to handle imbalance.

In [ ]:
# ===== TRAIN CATBOOST =====
cat_model = CatBoostClassifier(
    iterations=300,             # Number of boosting rounds
    depth=4,                    # Tree depth
    learning_rate=0.05,         # Small learning rate
    auto_class_weights='Balanced',   # Weight the minority class higher
    verbose=0,                  # Suppress training output
    random_state=RANDOM_STATE
)
cat_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_cat = cat_model.predict(X_test_scaled)
y_prob_cat = cat_model.predict_proba(X_test_scaled)[:, 1]

print("CatBoost — Test Results")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_cat):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_cat):.4f}")
print(f"F1 (Yes):  {f1_score(y_test, y_pred_cat):.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred_cat)}")

In [ ]:
# ===== CROSS-VALIDATE CATBOOST =====
cat_cv_scores = cross_val_report(
    CatBoostClassifier(iterations=300, depth=4, learning_rate=0.05,
                       auto_class_weights='Balanced', verbose=0, random_state=RANDOM_STATE),
    X_train_scaled, y_train, "CatBoost"
)

In [ ]:
# ===== CATBOOST CONFUSION MATRIX =====
fig, ax = plt.subplots(figsize=(6, 5))
cm_cat = confusion_matrix(y_test, y_pred_cat)
sns.heatmap(cm_cat, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=['Stay (0)', 'Leave (1)'],
            yticklabels=['Stay (0)', 'Leave (1)'])
ax.set_title('CatBoost — Confusion Matrix', fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# ===== CATBOOST FEATURE IMPORTANCE (Top 15) =====
feature_importance_cat = pd.DataFrame({
    'feature': X_train.columns,
    'importance': cat_model.feature_importances_
}).sort_values('importance', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feature_importance_cat['feature'], feature_importance_cat['importance'], color='#27ae60')
ax.set_title('CatBoost — Top 15 Feature Importances', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# ===== CATBOOST ROC CURVE =====
fpr_cat, tpr_cat, _ = roc_curve(y_test, y_prob_cat)
auc_cat = roc_auc_score(y_test, y_prob_cat)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_cat, tpr_cat, color='#27ae60', linewidth=2, label=f'CatBoost (AUC = {auc_cat:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_title('ROC Curve — CatBoost', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Neural Network (Custom Class Weights)

We add a **Deep Neural Network** to our comparison. Key features:
- **Custom class weights** to handle the imbalanced dataset (same ratio as XGBoost's `scale_pos_weight`)
- **Dropout regularization** to prevent overfitting on this small dataset
- **Early stopping** to find the optimal number of epochs automatically
- Architecture: 128 → 64 → 32 → 1 (sigmoid)

In [ ]:
# ===== NEURAL NETWORK WITH CUSTOM CLASS WEIGHTS =====
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TF = True
except ModuleNotFoundError:
    HAS_TF = False
    tf = None
    keras = None
    layers = None
    history = None
    nn_model = None
    y_prob_nn = None
    y_pred_nn = None
    fpr_nn = None
    tpr_nn = None
    auc_nn = None
    print("TensorFlow not installed; skipping neural network training.")

if HAS_TF:
    # Set seed for reproducibility
    tf.random.set_seed(RANDOM_STATE)

    # ===== CALCULATE CLASS WEIGHTS =====
    # Same logic as scale_pos_weight — give minority class more importance
    neg_count = (y_train == 0).sum()
    pos_count = (y_train == 1).sum()
    class_weight_dict = {
        0: 1.0,
        1: neg_count / pos_count  # ~5.2x weight for attrition class
    }
    print(f"Class weights: {{0: {class_weight_dict[0]:.2f}, 1: {class_weight_dict[1]:.2f}}}")
    print(f"→ Giving {class_weight_dict[1]:.1f}x more importance to 'Leave' samples")

    # ===== BUILD THE NEURAL NETWORK =====
    n_features = X_train_scaled.shape[1]

    nn_model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1, activation='sigmoid')
    ])

    nn_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    nn_model.summary()

    # ===== TRAIN WITH EARLY STOPPING =====
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    )

    history = nn_model.fit(
        X_train_scaled, y_train,
        epochs=100,
        batch_size=32,
        validation_split=0.2,
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=1
    )

In [ ]:
if not HAS_TF:
    print("TensorFlow not installed; skipping training curves.")
else:
    # ===== TRAINING CURVES =====
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history.history['loss'], label='Train Loss')
    ax1.plot(history.history['val_loss'], label='Val Loss')
    ax1.set_title('Neural Network — Loss Curve', fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history.history['accuracy'], label='Train Accuracy')
    ax2.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax2.set_title('Neural Network — Accuracy Curve', fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
if not HAS_TF:
    print("TensorFlow not installed; skipping neural network evaluation.")
else:
    # ===== NEURAL NETWORK EVALUATION =====
    y_prob_nn = nn_model.predict(X_test_scaled).flatten()
    y_pred_nn = (y_prob_nn >= 0.5).astype(int)

    print("Neural Network — Test Results")
    print("=" * 50)
    print(f"Accuracy:  {accuracy_score(y_test, y_pred_nn):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_nn):.4f}")
    print(f"F1 (Yes):  {f1_score(y_test, y_pred_nn):.4f}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred_nn)}")

In [ ]:
if not HAS_TF:
    print("TensorFlow not installed; skipping confusion matrix.")
else:
    # ===== NEURAL NETWORK CONFUSION MATRIX =====
    fig, ax = plt.subplots(figsize=(6, 5))
    cm_nn = confusion_matrix(y_test, y_pred_nn)
    sns.heatmap(cm_nn, annot=True, fmt='d', cmap='Purples', ax=ax,
                xticklabels=['Stay (0)', 'Leave (1)'],
                yticklabels=['Stay (0)', 'Leave (1)'])
    ax.set_title('Neural Network — Confusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.show()

In [ ]:
if not HAS_TF:
    print("TensorFlow not installed; skipping ROC curve.")
else:
    # ===== NEURAL NETWORK ROC CURVE =====
    fpr_nn, tpr_nn, _ = roc_curve(y_test, y_prob_nn)
    auc_nn = roc_auc_score(y_test, y_prob_nn)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(fpr_nn, tpr_nn, color='#9b59b6', linewidth=2, label=f'Neural Network (AUC = {auc_nn:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    ax.set_title('ROC Curve — Neural Network', fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Final Model Comparison

Now we compare all three models on the **same chart** and create a summary table  
to determine the best model. We focus on **F1-Score** and **ROC-AUC** — not just accuracy.

In [ ]:
# ===== COMBINED ROC CURVE — ALL 4 MODELS =====
fig, ax = plt.subplots(figsize=(8, 6))

# Logistic Regression
ax.plot(fpr_lr, tpr_lr, color='#3498db', linewidth=2.5,
        label=f'Logistic Regression (AUC = {auc_lr:.3f})')

# XGBoost
ax.plot(fpr_xgb, tpr_xgb, color='#e67e22', linewidth=2.5,
        label=f'XGBoost (AUC = {auc_xgb:.3f})')

# CatBoost
ax.plot(fpr_cat, tpr_cat, color='#27ae60', linewidth=2.5,
        label=f'CatBoost (AUC = {auc_cat:.3f})')

# Neural Network
if HAS_TF:
    ax.plot(fpr_nn, tpr_nn, color='#9b59b6', linewidth=2.5,
            label=f'Neural Network (AUC = {auc_nn:.3f})')
else:
    print("TensorFlow not installed; skipping neural network curve.")

# Random baseline
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random (AUC = 0.500)')

ax.set_title('ROC Curves — All Models Compared', fontweight='bold', fontsize=14)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ===== SUMMARY COMPARISON TABLE =====
from sklearn.metrics import precision_score, recall_score

results = []
models = {
    'Logistic Regression': (y_pred_lr, y_prob_lr),
    'XGBoost': (y_pred_xgb, y_prob_xgb),
    'CatBoost': (y_pred_cat, y_prob_cat),
}
if HAS_TF:
    models['Neural Network'] = (y_pred_nn, y_prob_nn)
else:
    print("TensorFlow not installed; skipping neural network in summary table.")

for name, (y_pred, y_prob) in models.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (Attrition)': precision_score(y_test, y_pred),
        'Recall (Attrition)': recall_score(y_test, y_pred),
        'F1 (Attrition)': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob),
    })

results_df = pd.DataFrame(results).set_index('Model')

# Format as percentages for readability
results_styled = results_df.style.format('{:.4f}').highlight_max(
    axis=0, color='lightgreen'
).set_caption('Model Comparison — Higher is Better (Green = Best)')

results_styled

In [ ]:
# ===== PRINT FINAL COMPARISON =====
print("\n" + "=" * 70)
print("FINAL MODEL COMPARISON \u2014 Test Set Metrics")
print("=" * 70)
print(results_df.to_string())

# Cross-Validation Summary
print("\n" + "=" * 70)
print("CROSS-VALIDATION RESULTS (5-Fold Stratified \u2014 More Reliable!)")
print("=" * 70)
cv_summary = pd.DataFrame({
    'Logistic Regression': lr_cv_scores,
    'XGBoost (Tuned)': xgb_cv_scores,
    'CatBoost': cat_cv_scores
}).T
cv_summary.columns = ['CV Accuracy', 'CV F1', 'CV ROC-AUC']
print(cv_summary.to_string())
print("=" * 70)

best_f1_model = results_df['F1 (Attrition)'].idxmax()
best_auc_model = results_df['ROC-AUC'].idxmax()

print(f"\n\U0001f3c6 Best model by F1 Score:  {best_f1_model} ({results_df.loc[best_f1_model, 'F1 (Attrition)']:.4f})")
print(f"\U0001f3c6 Best model by ROC-AUC:   {best_auc_model} ({results_df.loc[best_auc_model, 'ROC-AUC']:.4f})")
print(f"\n\U0001f4ca Cross-validation confirms these results are stable, not a lucky split!")

## Final Conclusion

### Why we don't judge by Accuracy alone:
- A dummy classifier gets ~84% accuracy by always predicting "No"
- **F1-Score** tells us how well we detect employees who actually leave
- **ROC-AUC** measures overall discriminative ability

### Model Selection Guidelines:
| If your priority is... | Use this model |
|---|---|
| **Catching as many leavers as possible** (high recall) | Model with highest Recall |
| **Balanced precision & recall** | Model with highest F1-Score |
| **Overall ranking ability** | Model with highest ROC-AUC |
| **Interpretability / explainability** | Logistic Regression |
| **Raw predictive power** | XGBoost or CatBoost |

### Business Recommendation:
For HR attrition prediction, **recall is often more important than precision** —  
it's better to flag an employee who might leave (false alarm) than to miss someone  
who actually leaves (missed detection). Use this to guide your model choice.



In [ ]:
# ===== SAVE ALL TRAINED MODELS =====
import joblib
import os

# Create a directory for saved models
MODELS_DIR = 'saved_models'
os.makedirs(MODELS_DIR, exist_ok=True)

# --- Save XGBoost model ---
xgb_path = os.path.join(MODELS_DIR, 'xgboost_best_model.pkl')
joblib.dump(xgb_model, xgb_path)
print(f'XGBoost model saved to: {xgb_path}')

# --- Save CatBoost model ---
cat_path = os.path.join(MODELS_DIR, 'catboost_model.pkl')
joblib.dump(cat_model, cat_path)
print(f'CatBoost model saved to: {cat_path}')

# --- Save Neural Network model (Keras) ---
if HAS_TF and nn_model is not None:
    nn_path = os.path.join(MODELS_DIR, 'neural_network_model.h5')
    nn_model.save(nn_path)
    print(f'Neural Network model saved to: {nn_path}')
else:
    print('Neural Network model not saved (TensorFlow not available).')

print('\n All models saved successfully!')
print(f'Models directory: {os.path.abspath(MODELS_DIR)}')


In [ ]:
!pip install mlflow

In [ ]:
# ===== MLFLOW MODEL TRACKING & EVALUATION =====
import mlflow
import mlflow.sklearn
import joblib
import os
import pandas as pd
from mlflow.models import infer_signature
import logging
logging.getLogger('mlflow').setLevel(logging.ERROR)


# Load the saved XGBoost model
xgb_model_loaded = joblib.load(os.path.join('saved_models', 'xgboost_best_model.pkl'))

# Use REAL feature names from training data (fixes feature_names mismatch)
feature_names = list(X_train.columns)

# Create evaluation DataFrame with correct column names
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)
eval_data = X_test_df.copy()
eval_data['label'] = y_test.values

with mlflow.start_run():
    # Log model with signature
    signature = infer_signature(X_test_df, xgb_model_loaded.predict(X_test_df))
    model_info = mlflow.sklearn.log_model(xgb_model_loaded, name='xgboost_best_model', signature=signature)

    # Evaluate
    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data,
        targets='label',
        model_type='classifier',
    )

    print(f"Accuracy: {result.metrics['accuracy_score']:.3f}")
    print(f"F1 Score: {result.metrics['f1_score']:.3f}")
    print(f"ROC AUC: {result.metrics['roc_auc']:.3f}")


In [ ]:
# ===== MLFLOW MODEL TRACKING & EVALUATION (CatBoost) =====
import mlflow
import mlflow.catboost
import joblib
import os
import pandas as pd
from mlflow.models import infer_signature
import logging
logging.getLogger('mlflow').setLevel(logging.ERROR)

# Load the saved CatBoost model
cat_model_loaded = joblib.load(os.path.join('saved_models', 'catboost_model.pkl'))

# Use REAL feature names from training data
feature_names = list(X_train.columns)

# Create evaluation DataFrame with correct column names
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)
eval_data = X_test_df.copy()
eval_data['label'] = y_test.values

with mlflow.start_run():
    # Log model with signature
    signature = infer_signature(X_test_df, cat_model_loaded.predict(X_test_df))
    model_info = mlflow.catboost.log_model(cat_model_loaded, artifact_path='catboost_model', signature=signature)

    # Evaluate
    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data,
        targets='label',
        model_type='classifier',
    )

    print(f"Accuracy: {result.metrics['accuracy_score']:.3f}")
    print(f"F1 Score: {result.metrics['f1_score']:.3f}")
    print(f"ROC AUC: {result.metrics['roc_auc']:.3f}")

